In [2]:
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

# load cleaned datasets
train_df = pd.read_csv("data/booking_reviews_train.csv")
test_df = pd.read_csv("data/booking_reviews_test.csv")

# features + labels
X_train_text = train_df["text_combined"].fillna("")
X_test_text = test_df["text_combined"].fillna("")

X_train_clean = train_df[["cleanliness_score"]].fillna(0)
X_test_clean = test_df[["cleanliness_score"]].fillna(0)

y_train = train_df["is_positive"]
y_test = test_df["is_positive"]

# hold out a validation split to learn ensemble weights
X_base_text, X_val_text, X_base_clean, X_val_clean, y_base, y_val = train_test_split(
    X_train_text,
    X_train_clean,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

def build_sparse_features(vectorizer, text_data, clean_data, fit=False):
    if fit:
        text_matrix = vectorizer.fit_transform(text_data)
    else:
        text_matrix = vectorizer.transform(text_data)
    return hstack([text_matrix, csr_matrix(clean_data.values)])

def print_metrics(model_name, y_true, y_pred):
    print(f"{model_name}:")
    print("  Accuracy:", round(accuracy_score(y_true, y_pred), 3))
    print("  Precision:", round(precision_score(y_true, y_pred), 3))
    print("  Recall:", round(recall_score(y_true, y_pred), 3))
    print("  F1 Score:", round(f1_score(y_true, y_pred), 3))

# validation-stage feature extraction
val_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
 )

X_base_sparse = build_sparse_features(val_vectorizer, X_base_text, X_base_clean, fit=True)
X_val_sparse = build_sparse_features(val_vectorizer, X_val_text, X_val_clean)

val_svd_components = max(2, min(300, X_base_sparse.shape[1] - 1))
val_svd = TruncatedSVD(n_components=val_svd_components, random_state=42)
X_base_nn = val_svd.fit_transform(X_base_sparse)
X_val_nn = val_svd.transform(X_val_sparse)

# base models used to estimate ensemble weights
val_log_reg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)
val_naive_bayes = MultinomialNB()
val_svm = CalibratedClassifierCV(
    LinearSVC(class_weight="balanced", random_state=42, max_iter=3000),
    cv=3
 )
val_neural_net = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    max_iter=50,
    early_stopping=True,
    random_state=42
 )

val_log_reg.fit(X_base_sparse, y_base)
val_naive_bayes.fit(X_base_sparse, y_base)
val_svm.fit(X_base_sparse, y_base)
val_neural_net.fit(X_base_nn, y_base)

val_proba_log_reg = val_log_reg.predict_proba(X_val_sparse)[:, 1]
val_proba_naive_bayes = val_naive_bayes.predict_proba(X_val_sparse)[:, 1]
val_proba_svm = val_svm.predict_proba(X_val_sparse)[:, 1]
val_proba_neural_net = val_neural_net.predict_proba(X_val_nn)[:, 1]

val_pred_log_reg = (val_proba_log_reg >= 0.5).astype(int)
val_pred_naive_bayes = (val_proba_naive_bayes >= 0.5).astype(int)
val_pred_svm = (val_proba_svm >= 0.5).astype(int)
val_pred_neural_net = (val_proba_neural_net >= 0.5).astype(int)

validation_scores = {
    "Logistic Regression": max(f1_score(y_val, val_pred_log_reg), 1e-6),
    "Naive Bayes": max(f1_score(y_val, val_pred_naive_bayes), 1e-6),
    "SVM": max(f1_score(y_val, val_pred_svm), 1e-6),
    "Neural Net": max(f1_score(y_val, val_pred_neural_net), 1e-6),
}

weight_total = sum(validation_scores.values())
model_weights = {name: score / weight_total for name, score in validation_scores.items()}

# refit on the full training set before final test evaluation
full_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
 )

X_train_sparse = build_sparse_features(full_vectorizer, X_train_text, X_train_clean, fit=True)
X_test_sparse = build_sparse_features(full_vectorizer, X_test_text, X_test_clean)

full_svd_components = max(2, min(300, X_train_sparse.shape[1] - 1))
full_svd = TruncatedSVD(n_components=full_svd_components, random_state=42)
X_train_nn = full_svd.fit_transform(X_train_sparse)
X_test_nn = full_svd.transform(X_test_sparse)

log_reg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)
naive_bayes = MultinomialNB()
svm = CalibratedClassifierCV(
    LinearSVC(class_weight="balanced", random_state=42, max_iter=3000),
    cv=3
 )
neural_net = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    max_iter=50,
    early_stopping=True,
    random_state=42
 )

log_reg.fit(X_train_sparse, y_train)
naive_bayes.fit(X_train_sparse, y_train)
svm.fit(X_train_sparse, y_train)
neural_net.fit(X_train_nn, y_train)

proba_log_reg = log_reg.predict_proba(X_test_sparse)[:, 1]
proba_naive_bayes = naive_bayes.predict_proba(X_test_sparse)[:, 1]
proba_svm = svm.predict_proba(X_test_sparse)[:, 1]
proba_neural_net = neural_net.predict_proba(X_test_nn)[:, 1]

pred_log_reg = (proba_log_reg >= 0.5).astype(int)
pred_naive_bayes = (proba_naive_bayes >= 0.5).astype(int)
pred_svm = (proba_svm >= 0.5).astype(int)
pred_neural_net = (proba_neural_net >= 0.5).astype(int)

weighted_score = (
    model_weights["Logistic Regression"] * proba_log_reg +
    model_weights["Naive Bayes"] * proba_naive_bayes +
    model_weights["SVM"] * proba_svm +
    model_weights["Neural Net"] * proba_neural_net
)
y_pred = (weighted_score >= 0.5).astype(int)

print("=== Validation-Derived Ensemble Weights ===")
for model_name, weight in model_weights.items():
    print(f"{model_name}: {weight:.3f}")

print("\n=== Individual Model Scores ===")
for model_name, preds in [
    ("Logistic Regression", pred_log_reg),
    ("Naive Bayes", pred_naive_bayes),
    ("SVM", pred_svm),
    ("Neural Net", pred_neural_net),
]:
    print_metrics(model_name, y_test, preds)

print("\n=== Weighted Ensemble Results ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1 Score:", round(f1_score(y_test, y_pred), 3))

print("\nReport:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

c:\Users\ritam\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\ritam\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\ritam\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\ritam\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\ritam\anaconda3\Lib\site-packages\sklearn\svm\_clas

=== Validation-Derived Ensemble Weights ===
Logistic Regression: 0.248
Naive Bayes: 0.252
SVM: 0.251
Neural Net: 0.250

=== Individual Model Scores ===
Logistic Regression:
  Accuracy: 0.842
  Precision: 0.954
  Recall: 0.824
  F1 Score: 0.885
Naive Bayes:
  Accuracy: 0.847
  Precision: 0.849
  Recall: 0.962
  F1 Score: 0.902
SVM:
  Accuracy: 0.85
  Precision: 0.896
  Recall: 0.9
  F1 Score: 0.898
Neural Net:
  Accuracy: 0.86
  Precision: 0.914
  Recall: 0.892
  F1 Score: 0.903

=== Weighted Ensemble Results ===
Accuracy: 0.86
Precision: 0.916
Recall: 0.891
F1 Score: 0.903

Report:
              precision    recall  f1-score   support

           0       0.72      0.77      0.75      1396
           1       0.92      0.89      0.90      3857

    accuracy                           0.86      5253
   macro avg       0.82      0.83      0.82      5253
weighted avg       0.86      0.86      0.86      5253

Confusion Matrix:
[[1079  317]
 [ 419 3438]]
